# Data Cleaning and Preprocessing

## Section 1: Kaggle Environment Setup

This section checks:

- Python version
- Kaggle environment
- GPU availability
- Memory and storage
- Random seed

A GPU is optional for data cleaning but may help with OCR.

In [1]:
import os
import sys
import random
import shutil
import subprocess
from pathlib import Path

import numpy as np
import psutil

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

# Kaggle environment
IS_KAGGLE = Path("/kaggle").exists()
INPUT_DIR = Path("/kaggle/input") if IS_KAGGLE else Path("input")
OUTPUT_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("output")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python version : {sys.version.split()[0]}")
print(f"Kaggle detected: {IS_KAGGLE}")
print(f"Input folder   : {INPUT_DIR}")
print(f"Output folder  : {OUTPUT_DIR}")
print(f"Random seed    : {RANDOM_SEED}")

Python version : 3.12.13
Kaggle detected: True
Input folder   : /kaggle/input
Output folder  : /kaggle/working
Random seed    : 42


In [2]:
# GPU check
try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total",
         "--format=csv,noheader"],
        capture_output=True,
        text=True,
        check=True,
    )
    print(f"GPU             : {result.stdout.strip()}")
except (FileNotFoundError, subprocess.CalledProcessError):
    print("GPU             : No NVIDIA GPU detected")

GPU             : Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


In [3]:
# Memory and storage check
GB = 1024 ** 3

memory = psutil.virtual_memory()
disk = shutil.disk_usage(OUTPUT_DIR)

print(f"Total memory    : {memory.total / GB:.2f} GB")
print(f"Available memory: {memory.available / GB:.2f} GB")
print(f"Free storage    : {disk.free / GB:.2f} GB")

Total memory    : 31.35 GB
Available memory: 30.17 GB
Free storage    : 19.50 GB


## Section 2: Install and Import Required Libraries

This section prepares the libraries needed for PDF extraction, OCR, text cleaning, duplicate detection, and dataset export.

In [4]:
%pip install -q pypdf pymupdf pdfplumber pytesseract rapidfuzz beautifulsoup4 lxml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 102.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.3.0 which is incompatible.
Note: you may need t

In [5]:
import re
import json
import hashlib
import shutil
from pathlib import Path

import fitz
import pandas as pd
import pdfplumber
import pytesseract

from bs4 import BeautifulSoup
from PIL import Image
from pypdf import PdfReader
from rapidfuzz import fuzz
from tqdm.auto import tqdm

print("All required Python libraries imported successfully.")

All required Python libraries imported successfully.


## Section 3: Configure Dataset Paths

This section locates the source documents and creates folders for processed outputs.

In [6]:
# Display attached Kaggle datasets
from pathlib import Path

INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/tax_llm")

dataset_folders = [path for path in INPUT_DIR.iterdir() if path.is_dir()]

print("Attached Kaggle datasets:")
for folder in dataset_folders:
    print(f"- {folder.name}")

Attached Kaggle datasets:
- datasets


In [7]:
# Create output folders
CLEANED_DIR = OUTPUT_DIR / "cleaned_documents"
REPORTS_DIR = OUTPUT_DIR / "reports"
EXPORT_DIR = OUTPUT_DIR / "exports"

for folder in [OUTPUT_DIR, CLEANED_DIR, REPORTS_DIR, EXPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Input folder   : {INPUT_DIR}")
print(f"Output folder  : {OUTPUT_DIR}")
print(f"Cleaned folder : {CLEANED_DIR}")
print(f"Reports folder : {REPORTS_DIR}")
print(f"Export folder  : {EXPORT_DIR}")

Input folder   : /kaggle/input
Output folder  : /kaggle/working/tax_llm
Cleaned folder : /kaggle/working/tax_llm/cleaned_documents
Reports folder : /kaggle/working/tax_llm/reports
Export folder  : /kaggle/working/tax_llm/exports


In [8]:
SUPPORTED_EXTENSIONS = {
    ".pdf",
    ".txt",
    ".html",
    ".htm",
    ".csv",
    ".json",
    ".jsonl",
}

source_files = [
    path
    for path in INPUT_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
]

print(f"Supported source files found: {len(source_files)}")

for path in source_files[:20]:
    print(path)

Supported source files found: 58
/kaggle/input/datasets/nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/forms/f1065x--2025.pdf
/kaggle/input/datasets/nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/forms/f1065--2025.pdf
/kaggle/input/datasets/nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/publications/p541--2025.pdf
/kaggle/input/datasets/nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/instructions/i1065x--2025.pdf
/kaggle/input/datasets/nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/instructions/i1065--2025.pdf
/kaggle/input/datasets/nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/schedule_instructions/i1065sd--2025.pdf
/kaggle/input/datasets/nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/schedule_instructions/i1065sb2--2018.pdf
/kaggle/input/datasets/nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/schedule_instructions/i1065s23--2025.pdf
/kaggle/input/datasets/nallagantisharath/tax-data

## Section 3: Configure Dataset Paths

This section locates the raw tax documents and creates folders for processed outputs.

In [9]:
from pathlib import Path

INPUT_DIR = Path("/kaggle/input")

dataset_folders = sorted(
    path for path in INPUT_DIR.iterdir() if path.is_dir()
)

print(f"Attached datasets: {len(dataset_folders)}")

for index, folder in enumerate(dataset_folders):
    print(f"{index}: {folder}")

Attached datasets: 1
0: /kaggle/input/datasets


In [10]:
if len(dataset_folders) != 1:
    raise ValueError(
        "Expected one attached dataset. Select the correct folder manually."
    )

RAW_DATA_DIR = dataset_folders[0]

print(f"Raw data folder: {RAW_DATA_DIR}")

Raw data folder: /kaggle/input/datasets


## Create output folders

In [11]:
OUTPUT_DIR = Path("/kaggle/working/tax_llm")

CLEANED_DIR = OUTPUT_DIR / "cleaned_documents"
REPORTS_DIR = OUTPUT_DIR / "reports"
EXPORTS_DIR = OUTPUT_DIR / "exports"

for folder in [CLEANED_DIR, REPORTS_DIR, EXPORTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Cleaned documents: {CLEANED_DIR}")
print(f"Reports          : {REPORTS_DIR}")
print(f"Exports          : {EXPORTS_DIR}")

Cleaned documents: /kaggle/working/tax_llm/cleaned_documents
Reports          : /kaggle/working/tax_llm/reports
Exports          : /kaggle/working/tax_llm/exports


## Section 4: Discover Source Files

This section searches the selected dataset folder for supported tax-document files.

In [12]:
SUPPORTED_EXTENSIONS = {
    ".pdf",
    ".txt",
    ".html",
    ".htm",
    ".csv",
    ".json",
    ".jsonl",
}

source_files = sorted(
    path
    for path in RAW_DATA_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
)

print(f"Source files found: {len(source_files)}")

if source_files:
    for path in source_files[:20]:
        print(f"- {path.relative_to(RAW_DATA_DIR)}")

    if len(source_files) > 20:
        print(f"... and {len(source_files) - 20} more files")
else:
    print("No supported files found. Check RAW_DATA_DIR and the dataset contents.")

Source files found: 58
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/forms/f1065--2025.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/forms/f1065x--2025.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/guides/p4163.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/guides/p4164.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/instructions/i1065--2025.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/instructions/i1065x--2025.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/publications/p541--2025.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/schedule_instructions/i1065s23--2025.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/schedule_instructions/i1065sb2--2018.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/schedule_instructions/i1065sd--2025.pdf
- nallagantisharath/tax-data2/TAX/1065_Data/1065_federal/2025/schedul

## Section 5: Create the Document Inventory

This section creates a table containing basic information about every discovered source file.

In [13]:
import hashlib
import mimetypes

import pandas as pd
from tqdm.auto import tqdm


inventory_records = []

for file_path in tqdm(source_files, desc="Creating inventory"):
    relative_path = file_path.relative_to(RAW_DATA_DIR)
    file_stats = file_path.stat()

    # Create a stable ID from the relative file path
    document_id = hashlib.sha256(
        str(relative_path).encode("utf-8")
    ).hexdigest()[:16]

    inventory_records.append(
        {
            "document_id": document_id,
            "file_name": file_path.name,
            "relative_path": str(relative_path),
            "file_extension": file_path.suffix.lower(),
            "mime_type": mimetypes.guess_type(file_path.name)[0],
            "file_size_bytes": file_stats.st_size,
            "file_size_mb": round(file_stats.st_size / (1024 ** 2), 3),
        }
    )

document_inventory = pd.DataFrame(inventory_records)

print(f"Documents inventoried: {len(document_inventory)}")
display(document_inventory.head())

Creating inventory:   0%|          | 0/58 [00:00<?, ?it/s]

Documents inventoried: 58


,document_id,file_name,relative_path,file_extension,mime_type,file_size_bytes,file_size_mb
0,d304db736face09f,f1065--2025.pdf,nallagantisharath/tax-data2/TAX/1065_Data/1065...,.pdf,application/pdf,334608,0.319
1,afade3aec3a942e0,f1065x--2025.pdf,nallagantisharath/tax-data2/TAX/1065_Data/1065...,.pdf,application/pdf,168645,0.161
2,d1956ddbb121d7f5,p4163.pdf,nallagantisharath/tax-data2/TAX/1065_Data/1065...,.pdf,application/pdf,1415699,1.350
3,caa77758d1eecb37,p4164.pdf,nallagantisharath/tax-data2/TAX/1065_Data/1065...,.pdf,application/pdf,6909012,6.589
4,e54829d5a97de257,i1065--2025.pdf,nallagantisharath/tax-data2/TAX/1065_Data/1065...,.pdf,application/pdf,820966,0.783


## Review the inventory

In [14]:
print("Files by extension:")
display(
    document_inventory["file_extension"]
    .value_counts()
    .rename_axis("file_extension")
    .reset_index(name="file_count")
)

print(f"Total dataset size: {document_inventory['file_size_mb'].sum():.2f} MB")
print(f"Empty files found : {(document_inventory['file_size_bytes'] == 0).sum()}")

Files by extension:


,file_extension,file_count
0,.pdf,56
1,.csv,2


Total dataset size: 29.74 MB
Empty files found : 0


## save the inventory

In [15]:
inventory_path = REPORTS_DIR / "document_inventory.csv"

document_inventory.to_csv(inventory_path, index=False)

print(f"Inventory saved: {inventory_path}")

Inventory saved: /kaggle/working/tax_llm/reports/document_inventory.csv


## Section 6: Detect Form 1065 and Form 1120 Documents

This section identifies whether each document relates to Form 1065, Form 1120, both forms, or neither form.

In [16]:
def get_identification_text(file_path, max_pdf_pages=3):
    """Read a small amount of text for form detection."""

    extension = file_path.suffix.lower()

    try:
        if extension == ".pdf":
            document = fitz.open(file_path)

            text = " ".join(
                document[page_number].get_text()
                for page_number in range(min(max_pdf_pages, len(document)))
            )

            document.close()
            return text

        if extension in {".txt", ".csv", ".json", ".jsonl"}:
            return file_path.read_text(
                encoding="utf-8",
                errors="ignore"
            )[:50_000]

        if extension in {".html", ".htm"}:
            html = file_path.read_text(
                encoding="utf-8",
                errors="ignore"
            )

            return BeautifulSoup(html, "lxml").get_text(" ")[:50_000]

    except Exception:
        return ""

    return ""

## Detect the return type

In [17]:
FORM_1065_PATTERN = re.compile(
    r"\bform[\s_-]*1065(?!\d)",
    re.IGNORECASE
)

FORM_1120_PATTERN = re.compile(
    r"\bform[\s_-]*1120(?![\s_-]?[a-z]|\d)",
    re.IGNORECASE
)


def detect_return_type(file_path):
    relative_path = str(
        file_path.relative_to(RAW_DATA_DIR)
    ).replace("_", " ").replace("-", " ")

    document_text = get_identification_text(file_path)

    searchable_text = f"{relative_path} {document_text[:50_000]}"

    has_1065 = bool(FORM_1065_PATTERN.search(searchable_text))
    has_1120 = bool(FORM_1120_PATTERN.search(searchable_text))

    if has_1065 and has_1120:
        return "both"
    elif has_1065:
        return "1065"
    elif has_1120:
        return "1120"
    else:
        return "unknown"

## Update the inventory

In [18]:
file_lookup = {
    str(path.relative_to(RAW_DATA_DIR)): path
    for path in source_files
}

detected_return_types = []

for relative_path in tqdm(
    document_inventory["relative_path"],
    desc="Detecting return types"
):
    file_path = file_lookup[relative_path]
    detected_return_types.append(
        detect_return_type(file_path)
    )

document_inventory["return_type"] = detected_return_types

display(
    document_inventory[
        ["file_name", "file_extension", "return_type"]
    ].head(20)
)

Detecting return types:   0%|          | 0/58 [00:00<?, ?it/s]

,file_name,file_extension,return_type
0,f1065--2025.pdf,.pdf,1065
1,f1065x--2025.pdf,.pdf,1065
2,p4163.pdf,.pdf,unknown
3,p4164.pdf,.pdf,unknown
4,i1065--2025.pdf,.pdf,1065
5,i1065x--2025.pdf,.pdf,1065
6,p541--2025.pdf,.pdf,1065
7,i1065s23--2025.pdf,.pdf,1065
8,i1065sb2--2018.pdf,.pdf,1065
9,i1065sd--2025.pdf,.pdf,1065


## Review and save results

In [19]:
return_type_summary = (
    document_inventory["return_type"]
    .value_counts(dropna=False)
    .rename_axis("return_type")
    .reset_index(name="document_count")
)

display(return_type_summary)

unknown_count = (
    document_inventory["return_type"] == "unknown"
).sum()

both_count = (
    document_inventory["return_type"] == "both"
).sum()

print(f"Unknown documents     : {unknown_count}")
print(f"Documents mentioning both forms: {both_count}")

inventory_path = REPORTS_DIR / "document_inventory.csv"
document_inventory.to_csv(inventory_path, index=False)

print(f"Updated inventory saved: {inventory_path}")

,return_type,document_count
0,1120,23
1,1065,19
2,unknown,12
3,both,4


Unknown documents     : 12
Documents mentioning both forms: 4
Updated inventory saved: /kaggle/working/tax_llm/reports/document_inventory.csv


## Section 7: Detect the Tax Year

This section detects the tax year using the file name and the first pages of each document.

## Tax year Detrection Function

In [20]:
from collections import Counter
from datetime import datetime


MIN_TAX_YEAR = 1990
MAX_TAX_YEAR = datetime.now().year + 1

YEAR_PATTERN = rf"\b(?:{MIN_TAX_YEAR}|19[9][0-9]|20[0-9]{{2}})\b"


def detect_tax_year(file_path):
    file_name_text = file_path.stem.replace("_", " ").replace("-", " ")
    document_text = get_identification_text(file_path, max_pdf_pages=3)

    year_scores = Counter()
    year_sources = {}

    # Years found in the file name receive higher priority
    filename_years = re.findall(YEAR_PATTERN, file_name_text)

    for year in filename_years:
        year_number = int(year)

        if MIN_TAX_YEAR <= year_number <= MAX_TAX_YEAR:
            year_scores[year_number] += 5
            year_sources.setdefault(year_number, set()).add("file_name")

    # Search for clear tax-year statements
    tax_year_patterns = [
        rf"\btax\s+year\s+({YEAR_PATTERN})",
        rf"\bcalendar\s+year\s+({YEAR_PATTERN})",
        rf"\b({YEAR_PATTERN})\s+instructions?\s+for\s+form\b",
        rf"\b({YEAR_PATTERN})\s+form\s+(?:1065|1120)\b",
        rf"\bform\s+(?:1065|1120)\s*\(?({YEAR_PATTERN})\)?",
    ]

    for pattern in tax_year_patterns:
        for match in re.findall(pattern, document_text, flags=re.IGNORECASE):
            # Nested regex groups may return tuples
            if isinstance(match, tuple):
                match = next(
                    value for value in match
                    if value and re.fullmatch(r"\d{4}", value)
                )

            year_number = int(match)

            if MIN_TAX_YEAR <= year_number <= MAX_TAX_YEAR:
                year_scores[year_number] += 3
                year_sources.setdefault(year_number, set()).add("document_text")

    if not year_scores:
        return {
            "tax_year": None,
            "tax_year_source": "not_detected",
            "tax_year_status": "review",
        }

    ranked_years = year_scores.most_common()
    best_year, best_score = ranked_years[0]

    # Flag equal scores for manual review
    tied_years = [
        year for year, score in ranked_years
        if score == best_score
    ]

    status = "detected" if len(tied_years) == 1 else "ambiguous"

    return {
        "tax_year": best_year,
        "tax_year_source": ", ".join(
            sorted(year_sources.get(best_year, {"unknown"}))
        ),
        "tax_year_status": status,
    }

In [21]:
# Update the inventory

tax_year_results = []

for relative_path in tqdm(
    document_inventory["relative_path"],
    desc="Detecting tax years"
):
    file_path = file_lookup[relative_path]
    tax_year_results.append(detect_tax_year(file_path))


tax_year_table = pd.DataFrame(tax_year_results)

document_inventory["tax_year"] = pd.array(
    tax_year_table["tax_year"],
    dtype="Int64"
)

document_inventory["tax_year_source"] = (
    tax_year_table["tax_year_source"]
)

document_inventory["tax_year_status"] = (
    tax_year_table["tax_year_status"]
)

display(
    document_inventory[
        [
            "file_name",
            "return_type",
            "tax_year",
            "tax_year_source",
            "tax_year_status",
        ]
    ].head(20)
)

Detecting tax years:   0%|          | 0/58 [00:00<?, ?it/s]

,file_name,return_type,tax_year,tax_year_source,tax_year_status
0,f1065--2025.pdf,1065,2025,"document_text, file_name",detected
1,f1065x--2025.pdf,1065,2025,file_name,detected
2,p4163.pdf,unknown,<NA>,not_detected,review
3,p4164.pdf,unknown,<NA>,not_detected,review
4,i1065--2025.pdf,1065,2025,"document_text, file_name",detected
5,i1065x--2025.pdf,1065,2025,file_name,detected
6,p541--2025.pdf,1065,2025,file_name,detected
7,i1065s23--2025.pdf,1065,2025,"document_text, file_name",detected
8,i1065sb2--2018.pdf,1065,2018,file_name,detected
9,i1065sd--2025.pdf,1065,2025,file_name,detected


In [22]:
# Review and save the results

tax_year_summary = (
    document_inventory
    .groupby(["tax_year", "return_type"], dropna=False)
    .size()
    .reset_index(name="document_count")
    .sort_values(["tax_year", "return_type"])
)

display(tax_year_summary)

review_documents = document_inventory[
    document_inventory["tax_year_status"] != "detected"
]

print(f"Detected tax years : {(document_inventory['tax_year_status'] == 'detected').sum()}")
print(f"Needs review       : {len(review_documents)}")

if not review_documents.empty:
    display(
        review_documents[
            ["file_name", "return_type", "tax_year", "tax_year_status"]
        ].head(20)
    )

inventory_path = REPORTS_DIR / "document_inventory.csv"
document_inventory.to_csv(inventory_path, index=False)

print(f"Updated inventory saved: {inventory_path}")

,tax_year,return_type,document_count
0,2011,1120,2
1,2014,1065,1
2,2015,1120,1
3,2016,1120,2
4,2016,unknown,1
5,2018,1065,2
6,2018,1120,4
7,2018,both,1
8,2019,1065,1
9,2019,1120,1


Detected tax years : 54
Needs review       : 4


,file_name,return_type,tax_year,tax_year_status
2,p4163.pdf,unknown,<NA>,review
3,p4164.pdf,unknown,<NA>,review
38,p4163.pdf,unknown,<NA>,review
39,p4164.pdf,unknown,<NA>,review


Updated inventory saved: /kaggle/working/tax_llm/reports/document_inventory.csv


## Section 8: Extract Text from Digital PDFs

This section extracts text from searchable PDF files and identifies PDFs that may require OCR.

## Create extraction folder

In [23]:
PDF_TEXT_DIR = CLEANED_DIR / "pdf_text"
PDF_TEXT_DIR.mkdir(parents=True, exist_ok=True)

MIN_PAGE_CHARACTERS = 50
MIN_DOCUMENT_CHARACTERS = 200

print(f"PDF text folder: {PDF_TEXT_DIR}")

PDF text folder: /kaggle/working/tax_llm/cleaned_documents/pdf_text


## Define the extraction function

In [24]:
def extract_pdf_text(file_path, document_id):
    output_path = PDF_TEXT_DIR / f"{document_id}.txt"

    try:
        page_texts = []

        with fitz.open(file_path) as pdf_document:
            page_count = len(pdf_document)

            for page_number, page in enumerate(pdf_document, start=1):
                text = page.get_text("text").strip()

                page_texts.append(
                    f"\n--- Page {page_number} ---\n{text}"
                )

        full_text = "\n".join(page_texts).strip()
        character_count = len(full_text)

        pages_with_text = sum(
            len(page_text.strip()) >= MIN_PAGE_CHARACTERS
            for page_text in page_texts
        )

        text_page_ratio = (
            pages_with_text / page_count
            if page_count > 0
            else 0
        )

        if (
            character_count >= MIN_DOCUMENT_CHARACTERS
            and text_page_ratio >= 0.50
        ):
            extraction_status = "digital_text_extracted"
        else:
            extraction_status = "needs_ocr"

        output_path.write_text(
            full_text,
            encoding="utf-8"
        )

        return {
            "page_count": page_count,
            "character_count": character_count,
            "pages_with_text": pages_with_text,
            "text_page_ratio": round(text_page_ratio, 3),
            "extraction_status": extraction_status,
            "text_path": str(output_path.relative_to(OUTPUT_DIR)),
            "extraction_error": None,
        }

    except Exception as error:
        return {
            "page_count": None,
            "character_count": 0,
            "pages_with_text": 0,
            "text_page_ratio": 0,
            "extraction_status": "failed",
            "text_path": None,
            "extraction_error": str(error),
        }

## Extract all PDFs

In [25]:
extraction_results = []

pdf_inventory = document_inventory[
    document_inventory["file_extension"] == ".pdf"
]

for row in tqdm(
    pdf_inventory.itertuples(index=False),
    total=len(pdf_inventory),
    desc="Extracting PDF text"
):
    file_path = file_lookup[row.relative_path]

    result = extract_pdf_text(
        file_path=file_path,
        document_id=row.document_id,
    )

    result["document_id"] = row.document_id
    extraction_results.append(result)

pdf_extraction_table = pd.DataFrame(extraction_results)

print(f"PDFs processed: {len(pdf_extraction_table)}")
display(pdf_extraction_table.head())

Extracting PDF text:   0%|          | 0/56 [00:00<?, ?it/s]

PDFs processed: 56


,page_count,character_count,pages_with_text,text_page_ratio,extraction_status,text_path,extraction_error,document_id
0,6,25687,6,1.000,digital_text_extracted,cleaned_documents/pdf_text/d304db736face09f.txt,None,d304db736face09f
1,4,9804,4,1.000,digital_text_extracted,cleaned_documents/pdf_text/afade3aec3a942e0.txt,None,afade3aec3a942e0
2,96,226522,96,1.000,digital_text_extracted,cleaned_documents/pdf_text/d1956ddbb121d7f5.txt,None,d1956ddbb121d7f5
3,282,474824,280,0.993,digital_text_extracted,cleaned_documents/pdf_text/caa77758d1eecb37.txt,None,caa77758d1eecb37
4,70,458426,70,1.000,digital_text_extracted,cleaned_documents/pdf_text/e54829d5a97de257.txt,None,e54829d5a97de257


## Add results to the inventory

In [26]:
extraction_columns = [
    "document_id",
    "page_count",
    "character_count",
    "pages_with_text",
    "text_page_ratio",
    "extraction_status",
    "text_path",
    "extraction_error",
]

document_inventory = document_inventory.drop(
    columns=extraction_columns[1:],
    errors="ignore",
)

document_inventory = document_inventory.merge(
    pdf_extraction_table[extraction_columns],
    on="document_id",
    how="left",
)

document_inventory["extraction_status"] = (
    document_inventory["extraction_status"]
    .fillna("not_processed")
)

display(
    document_inventory[
        [
            "file_name",
            "page_count",
            "character_count",
            "text_page_ratio",
            "extraction_status",
        ]
    ].head(20)
)

,file_name,page_count,character_count,text_page_ratio,extraction_status
0,f1065--2025.pdf,6.0,25687.0,1.000,digital_text_extracted
1,f1065x--2025.pdf,4.0,9804.0,1.000,digital_text_extracted
2,p4163.pdf,96.0,226522.0,1.000,digital_text_extracted
3,p4164.pdf,282.0,474824.0,0.993,digital_text_extracted
4,i1065--2025.pdf,70.0,458426.0,1.000,digital_text_extracted
5,i1065x--2025.pdf,12.0,64008.0,1.000,digital_text_extracted
6,p541--2025.pdf,33.0,162331.0,1.000,digital_text_extracted
7,i1065s23--2025.pdf,47.0,307743.0,1.000,digital_text_extracted
8,i1065sb2--2018.pdf,2.0,7694.0,1.000,digital_text_extracted
9,i1065sd--2025.pdf,6.0,35125.0,1.000,digital_text_extracted


## Review and save

In [27]:
display(
    document_inventory["extraction_status"]
    .value_counts(dropna=False)
    .rename_axis("extraction_status")
    .reset_index(name="document_count")
)

needs_ocr = document_inventory[
    document_inventory["extraction_status"] == "needs_ocr"
]

failed_extractions = document_inventory[
    document_inventory["extraction_status"] == "failed"
]

print(f"PDFs requiring OCR : {len(needs_ocr)}")
print(f"Failed extractions : {len(failed_extractions)}")

document_inventory.to_csv(
    REPORTS_DIR / "document_inventory.csv",
    index=False,
)

pdf_extraction_table.to_csv(
    REPORTS_DIR / "pdf_extraction_report.csv",
    index=False,
)

print("Extraction results saved successfully.")

,extraction_status,document_count
0,digital_text_extracted,56
1,not_processed,2


PDFs requiring OCR : 0
Failed extractions : 0
Extraction results saved successfully.


## Section 9: Apply OCR to Scanned PDFs

This section uses OCR to extract text from scanned PDFs that could not be read digitally.

OCR uses the CPU and may take longer than normal PDF extraction.

## Configure OCR

In [28]:
OCR_TEXT_DIR = CLEANED_DIR / "ocr_text"
OCR_TEXT_DIR.mkdir(parents=True, exist_ok=True)

OCR_DPI = 200
OCR_CONFIG = "--oem 3 --psm 3"

ocr_candidates = document_inventory[
    document_inventory["extraction_status"] == "needs_ocr"
].copy()

print(f"PDFs requiring OCR: {len(ocr_candidates)}")
print(f"OCR output folder : {OCR_TEXT_DIR}")

PDFs requiring OCR: 0
OCR output folder : /kaggle/working/tax_llm/cleaned_documents/ocr_text


## Define the OCR function

In [29]:
def extract_pdf_with_ocr(file_path, document_id):
    output_path = OCR_TEXT_DIR / f"{document_id}.txt"

    try:
        page_texts = []
        ocr_page_count = 0
        scale = OCR_DPI / 72

        with fitz.open(file_path) as pdf_document:
            page_count = len(pdf_document)

            for page_number, page in enumerate(pdf_document, start=1):
                embedded_text = page.get_text("text").strip()

                # Keep usable digital text in mixed PDFs
                if len(embedded_text) >= MIN_PAGE_CHARACTERS:
                    page_text = embedded_text
                else:
                    pixmap = page.get_pixmap(
                        matrix=fitz.Matrix(scale, scale),
                        colorspace=fitz.csRGB,
                        alpha=False,
                    )

                    image = Image.frombytes(
                        "RGB",
                        (pixmap.width, pixmap.height),
                        pixmap.samples,
                    )

                    page_text = pytesseract.image_to_string(
                        image,
                        lang="eng",
                        config=OCR_CONFIG,
                    ).strip()

                    image.close()
                    ocr_page_count += 1

                page_texts.append(
                    f"--- Page {page_number} ---\n{page_text}"
                )

        full_text = "\n\n".join(page_texts).strip()
        character_count = len(full_text)

        if character_count >= MIN_DOCUMENT_CHARACTERS:
            status = "ocr_extracted"
        else:
            status = "ocr_low_text"

        output_path.write_text(full_text, encoding="utf-8")

        return {
            "document_id": document_id,
            "ocr_status": status,
            "ocr_page_count": ocr_page_count,
            "ocr_character_count": character_count,
            "ocr_text_path": str(output_path.relative_to(OUTPUT_DIR)),
            "ocr_error": None,
        }

    except Exception as error:
        return {
            "document_id": document_id,
            "ocr_status": "ocr_failed",
            "ocr_page_count": 0,
            "ocr_character_count": 0,
            "ocr_text_path": None,
            "ocr_error": str(error),
        }

## Test OCR on one PDF

In [30]:
if ocr_candidates.empty:
    print("No PDFs require OCR.")
else:
    test_row = ocr_candidates.iloc[0]
    test_file = file_lookup[test_row["relative_path"]]

    test_result = extract_pdf_with_ocr(
        test_file,
        test_row["document_id"],
    )

    print(test_result)

No PDFs require OCR.


In [31]:
# Process all OCR candidates

ocr_results = []

for row in tqdm(
    ocr_candidates.itertuples(index=False),
    total=len(ocr_candidates),
    desc="Applying OCR",
):
    file_path = file_lookup[row.relative_path]

    result = extract_pdf_with_ocr(
        file_path=file_path,
        document_id=row.document_id,
    )

    ocr_results.append(result)

ocr_table = pd.DataFrame(ocr_results)

print(f"PDFs processed with OCR: {len(ocr_table)}")

if not ocr_table.empty:
    display(ocr_table.head())

Applying OCR: 0it [00:00, ?it/s]

PDFs processed with OCR: 0


In [32]:
## Update and save the inventory

if not ocr_table.empty:
    ocr_columns = [
        "ocr_status",
        "ocr_page_count",
        "ocr_character_count",
        "ocr_text_path",
        "ocr_error",
    ]

    for column in ocr_columns:
        if column not in document_inventory.columns:
            document_inventory[column] = None

    ocr_lookup = ocr_table.set_index("document_id")

    for document_id in ocr_lookup.index:
        mask = document_inventory["document_id"] == document_id

        for column in ocr_columns:
            document_inventory.loc[mask, column] = (
                ocr_lookup.loc[document_id, column]
            )

        status = ocr_lookup.loc[document_id, "ocr_status"]

        document_inventory.loc[mask, "extraction_status"] = status

        if status in {"ocr_extracted", "ocr_low_text"}:
            document_inventory.loc[mask, "text_path"] = (
                ocr_lookup.loc[document_id, "ocr_text_path"]
            )

    display(
        document_inventory["extraction_status"]
        .value_counts(dropna=False)
        .rename_axis("extraction_status")
        .reset_index(name="document_count")
    )

    ocr_table.to_csv(
        REPORTS_DIR / "ocr_extraction_report.csv",
        index=False,
    )

document_inventory.to_csv(
    REPORTS_DIR / "document_inventory.csv",
    index=False,
)

print("OCR results saved successfully.")

OCR results saved successfully.


## Section 10: Extract Text from Non-PDF Files

This section extracts text from TXT, HTML, CSV, JSON, and JSONL files.

In [33]:
NON_PDF_TEXT_DIR = CLEANED_DIR / "non_pdf_text"
NON_PDF_TEXT_DIR.mkdir(parents=True, exist_ok=True)

NON_PDF_EXTENSIONS = {
    ".txt",
    ".html",
    ".htm",
    ".csv",
    ".json",
    ".jsonl",
}

non_pdf_inventory = document_inventory[
    document_inventory["file_extension"].isin(NON_PDF_EXTENSIONS)
].copy()

print(f"Non-PDF files found: {len(non_pdf_inventory)}")
print(f"Output folder      : {NON_PDF_TEXT_DIR}")

Non-PDF files found: 2
Output folder      : /kaggle/working/tax_llm/cleaned_documents/non_pdf_text


## Configure extraction

In [34]:
NON_PDF_TEXT_DIR = CLEANED_DIR / "non_pdf_text"
NON_PDF_TEXT_DIR.mkdir(parents=True, exist_ok=True)

NON_PDF_EXTENSIONS = {
    ".txt",
    ".html",
    ".htm",
    ".csv",
    ".json",
    ".jsonl",
}

non_pdf_inventory = document_inventory[
    document_inventory["file_extension"].isin(NON_PDF_EXTENSIONS)
].copy()

print(f"Non-PDF files found: {len(non_pdf_inventory)}")
print(f"Output folder      : {NON_PDF_TEXT_DIR}")

Non-PDF files found: 2
Output folder      : /kaggle/working/tax_llm/cleaned_documents/non_pdf_text


## Define the extraction function

In [35]:
def extract_non_pdf_text(file_path, document_id):
    output_path = NON_PDF_TEXT_DIR / f"{document_id}.txt"

    try:
        raw_text = file_path.read_text(
            encoding="utf-8",
            errors="ignore",
        )

        if file_path.suffix.lower() in {".html", ".htm"}:
            text = BeautifulSoup(
                raw_text,
                "lxml",
            ).get_text(separator="\n")

        else:
            text = raw_text

        # Remove null characters and excessive blank lines
        text = text.replace("\x00", "")
        text = re.sub(r"\n{3,}", "\n\n", text).strip()

        character_count = len(text)

        status = (
            "text_extracted"
            if character_count >= MIN_DOCUMENT_CHARACTERS
            else "low_text"
        )

        output_path.write_text(text, encoding="utf-8")

        return {
            "document_id": document_id,
            "non_pdf_status": status,
            "non_pdf_character_count": character_count,
            "non_pdf_text_path": str(
                output_path.relative_to(OUTPUT_DIR)
            ),
            "non_pdf_error": None,
        }

    except Exception as error:
        return {
            "document_id": document_id,
            "non_pdf_status": "failed",
            "non_pdf_character_count": 0,
            "non_pdf_text_path": None,
            "non_pdf_error": str(error),
        }

## Process the files

In [36]:
non_pdf_results = []

for row in tqdm(
    non_pdf_inventory.itertuples(index=False),
    total=len(non_pdf_inventory),
    desc="Extracting non-PDF text",
):
    file_path = file_lookup[row.relative_path]

    result = extract_non_pdf_text(
        file_path=file_path,
        document_id=row.document_id,
    )

    non_pdf_results.append(result)

non_pdf_table = pd.DataFrame(non_pdf_results)

print(f"Non-PDF files processed: {len(non_pdf_table)}")

if not non_pdf_table.empty:
    display(non_pdf_table.head())

Extracting non-PDF text:   0%|          | 0/2 [00:00<?, ?it/s]

Non-PDF files processed: 2


,document_id,non_pdf_status,non_pdf_character_count,non_pdf_text_path,non_pdf_error
0,9a21200b067fa27b,text_extracted,2418,cleaned_documents/non_pdf_text/9a21200b067fa27...,None
1,243c634d6655d5f1,text_extracted,2577,cleaned_documents/non_pdf_text/243c634d6655d5f...,None


## Update the inventory

In [37]:
if not non_pdf_table.empty:
    result_lookup = non_pdf_table.set_index("document_id")

    for document_id, result in result_lookup.iterrows():
        mask = document_inventory["document_id"] == document_id

        document_inventory.loc[
            mask, "character_count"
        ] = result["non_pdf_character_count"]

        document_inventory.loc[
            mask, "extraction_status"
        ] = result["non_pdf_status"]

        document_inventory.loc[
            mask, "text_path"
        ] = result["non_pdf_text_path"]

        document_inventory.loc[
            mask, "extraction_error"
        ] = result["non_pdf_error"]

## Review and save

In [38]:
display(
    document_inventory["extraction_status"]
    .value_counts(dropna=False)
    .rename_axis("extraction_status")
    .reset_index(name="document_count")
)

failed_non_pdf = non_pdf_table[
    non_pdf_table["non_pdf_status"] == "failed"
] if not non_pdf_table.empty else pd.DataFrame()

print(f"Failed non-PDF extractions: {len(failed_non_pdf)}")

document_inventory.to_csv(
    REPORTS_DIR / "document_inventory.csv",
    index=False,
)

if not non_pdf_table.empty:
    non_pdf_table.to_csv(
        REPORTS_DIR / "non_pdf_extraction_report.csv",
        index=False,
    )

print("Non-PDF extraction results saved successfully.")

,extraction_status,document_count
0,digital_text_extracted,56
1,text_extracted,2


Failed non-PDF extractions: 0
Non-PDF extraction results saved successfully.


## Section 11: Clean and Normalize Extracted Text


This section removes unwanted characters and normalizes spacing while preserving page boundaries and tax-document structure.

## Create the cleaned-text folder

In [39]:
import unicodedata

NORMALIZED_TEXT_DIR = CLEANED_DIR / "normalized_text"
NORMALIZED_TEXT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Normalized text folder: {NORMALIZED_TEXT_DIR}")

Normalized text folder: /kaggle/working/tax_llm/cleaned_documents/normalized_text


## Define the cleaning function

In [40]:
def clean_extracted_text(text):
    # Normalize Unicode characters
    text = unicodedata.normalize("NFKC", text)

    # Remove null bytes and control characters
    text = text.replace("\x00", "")
    text = re.sub(r"[\x01-\x08\x0b\x0c\x0e-\x1f\x7f]", "", text)

    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Join words split across lines by PDF extraction
    text = re.sub(
        r"(?<=[a-z])-\n(?=[a-z])",
        "",
        text,
    )

    # Replace repeated spaces and tabs without removing line breaks
    text = re.sub(r"[ \t]+", " ", text)

    # Remove spaces around line breaks
    text = re.sub(r" *\n *", "\n", text)

    # Limit excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

## Clean all extracted documents

In [41]:
cleaning_results = []

documents_with_text = document_inventory[
    document_inventory["text_path"].notna()
].copy()

for row in tqdm(
    documents_with_text.itertuples(index=False),
    total=len(documents_with_text),
    desc="Cleaning extracted text",
):
    try:
        source_text_path = OUTPUT_DIR / row.text_path
        normalized_path = NORMALIZED_TEXT_DIR / f"{row.document_id}.txt"

        raw_text = source_text_path.read_text(
            encoding="utf-8",
            errors="ignore",
        )

        cleaned_text = clean_extracted_text(raw_text)
        normalized_path.write_text(cleaned_text, encoding="utf-8")

        cleaning_results.append(
            {
                "document_id": row.document_id,
                "raw_character_count": len(raw_text),
                "clean_character_count": len(cleaned_text),
                "clean_text_path": str(
                    normalized_path.relative_to(OUTPUT_DIR)
                ),
                "cleaning_status": (
                    "cleaned"
                    if len(cleaned_text) >= MIN_DOCUMENT_CHARACTERS
                    else "low_text"
                ),
                "cleaning_error": None,
            }
        )

    except Exception as error:
        cleaning_results.append(
            {
                "document_id": row.document_id,
                "raw_character_count": 0,
                "clean_character_count": 0,
                "clean_text_path": None,
                "cleaning_status": "failed",
                "cleaning_error": str(error),
            }
        )

cleaning_table = pd.DataFrame(cleaning_results)

print(f"Documents processed: {len(cleaning_table)}")

if not cleaning_table.empty:
    display(cleaning_table.head())

Cleaning extracted text:   0%|          | 0/58 [00:00<?, ?it/s]

Documents processed: 58


,document_id,raw_character_count,clean_character_count,clean_text_path,cleaning_status,cleaning_error
0,d304db736face09f,25687,24921,cleaned_documents/normalized_text/d304db736fac...,cleaned,None
1,afade3aec3a942e0,9804,9629,cleaned_documents/normalized_text/afade3aec3a9...,cleaned,None
2,d1956ddbb121d7f5,226522,221609,cleaned_documents/normalized_text/d1956ddbb121...,cleaned,None
3,caa77758d1eecb37,474824,462892,cleaned_documents/normalized_text/caa77758d1ee...,cleaned,None
4,e54829d5a97de257,458426,452312,cleaned_documents/normalized_text/e54829d5a97d...,cleaned,None


## Update the inventory

In [42]:
cleaning_columns = [
    "document_id",
    "raw_character_count",
    "clean_character_count",
    "clean_text_path",
    "cleaning_status",
    "cleaning_error",
]

document_inventory = document_inventory.drop(
    columns=cleaning_columns[1:],
    errors="ignore",
)

if not cleaning_table.empty:
    document_inventory = document_inventory.merge(
        cleaning_table[cleaning_columns],
        on="document_id",
        how="left",
    )

document_inventory["cleaning_status"] = (
    document_inventory["cleaning_status"]
    .fillna("not_processed")
)

## Review and save

In [43]:
display(
    document_inventory["cleaning_status"]
    .value_counts(dropna=False)
    .rename_axis("cleaning_status")
    .reset_index(name="document_count")
)

failed_cleaning = document_inventory[
    document_inventory["cleaning_status"] == "failed"
]

low_text_cleaning = document_inventory[
    document_inventory["cleaning_status"] == "low_text"
]

print(f"Cleaning failures : {len(failed_cleaning)}")
print(f"Low-text documents: {len(low_text_cleaning)}")

document_inventory.to_csv(
    REPORTS_DIR / "document_inventory.csv",
    index=False,
)

cleaning_table.to_csv(
    REPORTS_DIR / "text_cleaning_report.csv",
    index=False,
)

print("Cleaned text and reports saved successfully.")

,cleaning_status,document_count
0,cleaned,58


Cleaning failures : 0
Low-text documents: 0
Cleaned text and reports saved successfully.


## Section 12: Detect Exact and Near-Duplicate Documents

This section detects identical and highly similar documents to prevent duplicate training data.

In [44]:
def normalize_for_duplicate_check(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


duplicate_records = []
document_texts = {}

cleaned_documents = document_inventory[
    document_inventory["clean_text_path"].notna()
].copy()

for row in tqdm(
    cleaned_documents.itertuples(index=False),
    total=len(cleaned_documents),
    desc="Calculating document hashes",
):
    try:
        text_path = OUTPUT_DIR / row.clean_text_path
        text = text_path.read_text(
            encoding="utf-8",
            errors="ignore",
        )

        normalized_text = normalize_for_duplicate_check(text)
        document_texts[row.document_id] = normalized_text

        text_hash = hashlib.sha256(
            normalized_text.encode("utf-8")
        ).hexdigest()

        duplicate_records.append(
            {
                "document_id": row.document_id,
                "text_hash": text_hash,
                "duplicate_error": None,
            }
        )

    except Exception as error:
        duplicate_records.append(
            {
                "document_id": row.document_id,
                "text_hash": None,
                "duplicate_error": str(error),
            }
        )

hash_table = pd.DataFrame(duplicate_records)

print(f"Documents hashed: {len(hash_table)}")
display(hash_table.head())

Calculating document hashes:   0%|          | 0/58 [00:00<?, ?it/s]

Documents hashed: 58


,document_id,text_hash,duplicate_error
0,d304db736face09f,65e1f8c96ccd034972267865827673884fd05e005dbeb4...,None
1,afade3aec3a942e0,a7295a055533aa0a7829be5fd755e18a92f3c9697e0dae...,None
2,d1956ddbb121d7f5,e93ae09b1cb015e0bcdc1849665aff703943fa1a95c9d4...,None
3,caa77758d1eecb37,1108a2e749f938675c7f57ff13af43712826e72f0565af...,None
4,e54829d5a97de257,65ae7eecc1409dc3f89dc381b6db6f21e2d1ac4db2432e...,None


## Detect exact duplicates

In [45]:
valid_hashes = hash_table[
    hash_table["text_hash"].notna()
].copy()

valid_hashes["exact_duplicate_rank"] = (
    valid_hashes.groupby("text_hash").cumcount()
)

valid_hashes["exact_duplicate"] = (
    valid_hashes["exact_duplicate_rank"] > 0
)

canonical_lookup = (
    valid_hashes.groupby("text_hash")["document_id"]
    .first()
    .to_dict()
)

valid_hashes["duplicate_of"] = valid_hashes.apply(
    lambda row: (
        canonical_lookup[row["text_hash"]]
        if row["exact_duplicate"]
        else None
    ),
    axis=1,
)

exact_duplicate_count = valid_hashes["exact_duplicate"].sum()

print(f"Exact duplicates found: {exact_duplicate_count}")

display(
    valid_hashes[
        valid_hashes["exact_duplicate"]
    ][
        ["document_id", "duplicate_of", "text_hash"]
    ].head(20)
)

Exact duplicates found: 2


,document_id,duplicate_of,text_hash
38,687f60f4af63f749,d1956ddbb121d7f5,e93ae09b1cb015e0bcdc1849665aff703943fa1a95c9d4...
39,f44c63cc48a3be4d,caa77758d1eecb37,1108a2e749f938675c7f57ff13af43712826e72f0565af...


## Detect near duplicates

In [46]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors


# Compare only canonical documents with enough text
canonical_documents = valid_hashes[
    ~valid_hashes["exact_duplicate"]
]["document_id"].tolist()

candidate_ids = [
    document_id
    for document_id in canonical_documents
    if len(document_texts.get(document_id, "")) >= MIN_DOCUMENT_CHARACTERS
]

candidate_texts = [
    document_texts[document_id]
    for document_id in candidate_ids
]

NEAR_DUPLICATE_THRESHOLD = 0.97
near_duplicate_records = []

if len(candidate_texts) >= 2:
    vectorizer = TfidfVectorizer(
        lowercase=False,
        analyzer="word",
        ngram_range=(1, 2),
        min_df=1,
        max_features=100_000,
        sublinear_tf=True,
    )

    text_vectors = vectorizer.fit_transform(candidate_texts)

    neighbor_count = min(5, len(candidate_ids))

    neighbor_model = NearestNeighbors(
        n_neighbors=neighbor_count,
        metric="cosine",
        algorithm="brute",
    )

    neighbor_model.fit(text_vectors)

    distances, indices = neighbor_model.kneighbors(text_vectors)

    compared_pairs = set()

    for source_index, document_id in enumerate(candidate_ids):
        for distance, target_index in zip(
            distances[source_index],
            indices[source_index],
        ):
            target_id = candidate_ids[target_index]

            if document_id == target_id:
                continue

            pair = tuple(sorted([document_id, target_id]))

            if pair in compared_pairs:
                continue

            compared_pairs.add(pair)
            similarity = 1 - float(distance)

            if similarity >= NEAR_DUPLICATE_THRESHOLD:
                near_duplicate_records.append(
                    {
                        "document_id_1": pair[0],
                        "document_id_2": pair[1],
                        "similarity_score": round(similarity, 4),
                        "review_status": "needs_review",
                    }
                )

near_duplicate_table = pd.DataFrame(
    near_duplicate_records,
    columns=[
        "document_id_1",
        "document_id_2",
        "similarity_score",
        "review_status",
    ],
)

print(f"Near-duplicate pairs found: {len(near_duplicate_table)}")

if not near_duplicate_table.empty:
    display(
        near_duplicate_table.sort_values(
            "similarity_score",
            ascending=False,
        ).head(20)
    )

Near-duplicate pairs found: 0


## Update the inventory

In [47]:
duplicate_columns = [
    "text_hash",
    "exact_duplicate",
    "duplicate_of",
]

document_inventory = document_inventory.drop(
    columns=duplicate_columns,
    errors="ignore",
)

document_inventory = document_inventory.merge(
    valid_hashes[
        [
            "document_id",
            "text_hash",
            "exact_duplicate",
            "duplicate_of",
        ]
    ],
    on="document_id",
    how="left",
)

document_inventory["exact_duplicate"] = (
    document_inventory["exact_duplicate"]
    .fillna(False)
    .astype(bool)
)

document_inventory["duplicate_status"] = "unique"

document_inventory.loc[
    document_inventory["exact_duplicate"],
    "duplicate_status",
] = "exact_duplicate"

# Near duplicates require manual review before removal
if not near_duplicate_table.empty:
    near_duplicate_ids = set(
        near_duplicate_table["document_id_1"]
    ) | set(
        near_duplicate_table["document_id_2"]
    )

    near_duplicate_mask = (
        document_inventory["document_id"].isin(near_duplicate_ids)
        & ~document_inventory["exact_duplicate"]
    )

    document_inventory.loc[
        near_duplicate_mask,
        "duplicate_status",
    ] = "possible_near_duplicate"

## Review and save

In [48]:
display(
    document_inventory["duplicate_status"]
    .value_counts(dropna=False)
    .rename_axis("duplicate_status")
    .reset_index(name="document_count")
)

document_inventory.to_csv(
    REPORTS_DIR / "document_inventory.csv",
    index=False,
)

valid_hashes.to_csv(
    REPORTS_DIR / "exact_duplicate_report.csv",
    index=False,
)

near_duplicate_table.to_csv(
    REPORTS_DIR / "near_duplicate_report.csv",
    index=False,
)

print("Duplicate reports saved successfully.")

,duplicate_status,document_count
0,unique,56
1,exact_duplicate,2


Duplicate reports saved successfully.


## Section 13: Validate Document Quality

This section calculates text-quality metrics, assigns a quality score, and identifies documents that require review or exclusion.

### Configure quality thresholds

In [49]:
MIN_QUALITY_CHARACTERS = 200
MIN_QUALITY_WORDS = 50
MIN_ALPHABETIC_RATIO = 0.45
MAX_REPLACEMENT_RATIO = 0.01

QUALITY_SCORE_REVIEW_THRESHOLD = 60
QUALITY_SCORE_GOOD_THRESHOLD = 80

### Define the quality-check function

In [50]:
def calculate_text_quality(text):
    character_count = len(text)
    word_count = len(re.findall(r"\b\w+\b", text))

    alphabetic_count = sum(
        character.isalpha()
        for character in text
    )

    printable_count = sum(
        character.isprintable() or character in "\n\t"
        for character in text
    )

    replacement_count = text.count("\ufffd")

    alphabetic_ratio = (
        alphabetic_count / character_count
        if character_count
        else 0
    )

    printable_ratio = (
        printable_count / character_count
        if character_count
        else 0
    )

    replacement_ratio = (
        replacement_count / character_count
        if character_count
        else 0
    )

    return {
        "quality_character_count": character_count,
        "quality_word_count": word_count,
        "alphabetic_ratio": round(alphabetic_ratio, 4),
        "printable_ratio": round(printable_ratio, 4),
        "replacement_character_ratio": round(
            replacement_ratio,
            4,
        ),
    }

### Score each document

In [51]:
def score_document_quality(inventory_row, text):
    metrics = calculate_text_quality(text)

    score = 100
    flags = []

    character_count = metrics["quality_character_count"]
    word_count = metrics["quality_word_count"]
    alphabetic_ratio = metrics["alphabetic_ratio"]
    printable_ratio = metrics["printable_ratio"]
    replacement_ratio = metrics["replacement_character_ratio"]

    if character_count < MIN_QUALITY_CHARACTERS:
        score -= 40
        flags.append("insufficient_text")
    elif character_count < 1_000:
        score -= 15
        flags.append("short_document")

    if word_count < MIN_QUALITY_WORDS:
        score -= 20
        flags.append("low_word_count")

    if alphabetic_ratio < MIN_ALPHABETIC_RATIO:
        score -= 20
        flags.append("low_alphabetic_ratio")

    if printable_ratio < 0.95:
        score -= 15
        flags.append("non_printable_characters")

    if replacement_ratio > MAX_REPLACEMENT_RATIO:
        score -= 20
        flags.append("encoding_or_ocr_noise")

    extraction_status = str(
        inventory_row.get("extraction_status", "")
    )

    cleaning_status = str(
        inventory_row.get("cleaning_status", "")
    )

    duplicate_status = str(
        inventory_row.get("duplicate_status", "")
    )

    return_type = str(
        inventory_row.get("return_type", "")
    )

    tax_year_status = str(
        inventory_row.get("tax_year_status", "")
    )

    if extraction_status in {
        "failed",
        "ocr_failed",
        "ocr_low_text",
        "low_text",
    }:
        score -= 25
        flags.append(f"extraction_{extraction_status}")

    if cleaning_status == "failed":
        score -= 30
        flags.append("cleaning_failed")

    if return_type in {"unknown", "both", "", "nan"}:
        score -= 10
        flags.append("return_type_review")

    if tax_year_status != "detected":
        score -= 10
        flags.append("tax_year_review")

    if duplicate_status == "exact_duplicate":
        score = 0
        flags.append("exact_duplicate")

    elif duplicate_status == "possible_near_duplicate":
        score -= 5
        flags.append("possible_near_duplicate")

    score = max(0, min(100, score))

    if score >= QUALITY_SCORE_GOOD_THRESHOLD:
        quality_grade = "good"
    elif score >= QUALITY_SCORE_REVIEW_THRESHOLD:
        quality_grade = "acceptable"
    else:
        quality_grade = "poor"

    mandatory_exclusion_flags = {
        "insufficient_text",
        "cleaning_failed",
        "exact_duplicate",
        "extraction_failed",
        "extraction_ocr_failed",
    }

    if mandatory_exclusion_flags.intersection(flags):
        training_status = "exclude"
    elif quality_grade == "poor" or flags:
        training_status = "review"
    else:
        training_status = "eligible"

    return {
        **metrics,
        "quality_score": score,
        "quality_grade": quality_grade,
        "quality_flags": (
            ";".join(sorted(set(flags)))
            if flags
            else "none"
        ),
        "training_status": training_status,
    }

### Validate all cleaned documents

In [52]:
quality_results = []

for _, row in tqdm(
    document_inventory.iterrows(),
    total=len(document_inventory),
    desc="Validating document quality",
):
    document_id = row["document_id"]
    clean_text_path = row.get("clean_text_path")

    if pd.isna(clean_text_path):
        quality_results.append(
            {
                "document_id": document_id,
                "quality_character_count": 0,
                "quality_word_count": 0,
                "alphabetic_ratio": 0,
                "printable_ratio": 0,
                "replacement_character_ratio": 0,
                "quality_score": 0,
                "quality_grade": "poor",
                "quality_flags": "missing_clean_text",
                "training_status": "exclude",
                "quality_error": None,
            }
        )
        continue

    try:
        text_file = OUTPUT_DIR / clean_text_path

        cleaned_text = text_file.read_text(
            encoding="utf-8",
            errors="replace",
        )

        result = score_document_quality(
            inventory_row=row,
            text=cleaned_text,
        )

        result["document_id"] = document_id
        result["quality_error"] = None
        quality_results.append(result)

    except Exception as error:
        quality_results.append(
            {
                "document_id": document_id,
                "quality_character_count": 0,
                "quality_word_count": 0,
                "alphabetic_ratio": 0,
                "printable_ratio": 0,
                "replacement_character_ratio": 0,
                "quality_score": 0,
                "quality_grade": "poor",
                "quality_flags": "quality_check_failed",
                "training_status": "exclude",
                "quality_error": str(error),
            }
        )

quality_table = pd.DataFrame(quality_results)

print(f"Documents validated: {len(quality_table)}")
display(quality_table.head())

Validating document quality:   0%|          | 0/58 [00:00<?, ?it/s]

Documents validated: 58


,quality_character_count,quality_word_count,alphabetic_ratio,printable_ratio,replacement_character_ratio,quality_score,quality_grade,quality_flags,training_status,document_id,quality_error
0,24921,3447,0.6106,1.0000,0.0,100,good,none,eligible,d304db736face09f,None
1,9629,1475,0.6510,1.0000,0.0,100,good,none,eligible,afade3aec3a942e0,None
2,221609,33941,0.7027,1.0000,0.0,80,good,return_type_review;tax_year_review,review,d1956ddbb121d7f5,None
3,462892,71729,0.7091,0.9999,0.0,80,good,return_type_review;tax_year_review,review,caa77758d1eecb37,None
4,452312,73817,0.7732,1.0000,0.0,100,good,none,eligible,e54829d5a97de257,None
